In [4]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed"

demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2003_2014_final.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2003_2014_appended.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_Questionnair_2003_2014_appended.csv"

# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file)
labs = pd.read_csv(labs_file)
questionnaire = pd.read_csv(questionnaire_file)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================

demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================

summary = []

for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)

display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

# =================================================================================================
# 🧠 Step 7: Create Domain Presence Flags
# =================================================================================================

demo["PRESENT_DEMOGRAPHICS"] = True
labs["PRESENT_LABS"] = True
questionnaire["PRESENT_QUESTIONNAIRE"] = True

# =================================================================================================
# 🔗 Step 8: Merge All Files Using FULL OUTER JOIN on SEQN
# =================================================================================================

df = demo.merge(
    labs,
    on="SEQN",
    how="outer",
    suffixes=("", "_LAB")
)

df = df.merge(
    questionnaire,
    on="SEQN",
    how="outer",
    suffixes=("", "_QUESTIONNAIRE")
)

# =================================================================================================
# 🧠 Step 9: Fill Presence Flags
# =================================================================================================

presence_cols = [
    "PRESENT_DEMOGRAPHICS",
    "PRESENT_LABS",
    "PRESENT_QUESTIONNAIRE"
]

for col in presence_cols:
    if col in df.columns:
        df[col] = df[col].fillna(False)

# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", df.shape)
print("Unique SEQN Count:", df["SEQN"].nunique())
print("Duplicate SEQN Count:", df.duplicated(subset=["SEQN"]).sum())

display(df.head(20))

# =================================================================================================
# 📊 Step 11: Data Availability Matrix
# =================================================================================================

availability_matrix = df[
    [
        "SEQN",
        "PRESENT_DEMOGRAPHICS",
        "PRESENT_LABS",
        "PRESENT_QUESTIONNAIRE"
    ]
].copy()

display(availability_matrix.head(20))

availability_summary = pd.DataFrame([
    {
        "domain": "Demographics",
        "available_records": df["PRESENT_DEMOGRAPHICS"].sum(),
        "missing_records": df.shape[0] - df["PRESENT_DEMOGRAPHICS"].sum(),
        "availability_percent": round((df["PRESENT_DEMOGRAPHICS"].sum() / df.shape[0]) * 100, 2)
    },
    {
        "domain": "Labs",
        "available_records": df["PRESENT_LABS"].sum(),
        "missing_records": df.shape[0] - df["PRESENT_LABS"].sum(),
        "availability_percent": round((df["PRESENT_LABS"].sum() / df.shape[0]) * 100, 2)
    },
    {
        "domain": "Questionnaire",
        "available_records": df["PRESENT_QUESTIONNAIRE"].sum(),
        "missing_records": df.shape[0] - df["PRESENT_QUESTIONNAIRE"].sum(),
        "availability_percent": round((df["PRESENT_QUESTIONNAIRE"].sum() / df.shape[0]) * 100, 2)
    }
])

display(availability_summary)

# =================================================================================================
# 🧪 Step 12: Optional Check - Rows Available in All Three Domains
# =================================================================================================

df["PRESENT_ALL_THREE_DOMAINS"] = (
    df["PRESENT_DEMOGRAPHICS"] &
    df["PRESENT_LABS"] &
    df["PRESENT_QUESTIONNAIRE"]
)

print("Records present in all three domains:", df["PRESENT_ALL_THREE_DOMAINS"].sum())

# =================================================================================================
# 💾 Step 13: Save Final Unified Dataset
# =================================================================================================

OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2003_2014.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)

# =================================================================================================
# 💾 Step 14: Save Availability Summary
# =================================================================================================

availability_summary_path = OUTPUT_DIR / "neighbourhood_nhanes_unified_2003_2014_availability_summary.csv"
availability_summary.to_csv(availability_summary_path, index=False)

print("\n✅ Availability summary saved to:")
print(availability_summary_path)

C:\Users\NVK00\AppData\Local\Temp\ipykernel_17832\3441062603.py:37: DtypeWarning: Columns (0: present_hdl) have mixed types. Specify dtype option on import or set low_memory=False.
  labs = pd.read_csv(labs_file)


📊 Loaded Files:
Demographics: (61087, 65)
Labs: (56139, 46)
Questionnaire: (61087, 333)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire


C:\Users\NVK00\AppData\Local\Temp\ipykernel_17832\3441062603.py:38: DtypeWarning: Columns (0: SMDUPCA) have mixed types. Specify dtype option on import or set low_memory=False.
  questionnaire = pd.read_csv(questionnaire_file)


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,61087,65,61087,0
1,labs,56139,46,56139,0
2,questionnaire,61087,333,61087,0


C:\Users\NVK00\AppData\Local\Temp\ipykernel_17832\3441062603.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  questionnaire["PRESENT_QUESTIONNAIRE"] = True


...

,SEQN,PRESENT_DEMOGRAPHICS,PRESENT_LABS,PRESENT_QUESTIONNAIRE
0,21005,True,True,True
1,21006,True,True,True
2,21007,True,True,True
3,21008,True,True,True
4,21009,True,True,True
5,21010,True,True,True
6,21011,True,False,True
7,21012,True,True,True
8,21013,True,True,True
9,21014,True,True,True


,domain,available_records,missing_records,availability_percent
0,Demographics,61087,0,100.0
1,Labs,56139,4948,91.9
2,Questionnaire,61087,0,100.0


Records present in all three domains: 56139


C:\Users\NVK00\AppData\Local\Temp\ipykernel_17832\3441062603.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["PRESENT_ALL_THREE_DOMAINS"] = (



✅ Final unified NHANES dataset saved to:
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_nhanes_unified_2003_2014.csv

✅ Availability summary saved to:
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_nhanes_unified_2003_2014_availability_summary.csv
